# The Electric Field — Simulations for Halliday, Resnick & Krane, Ch. 26

This notebook follows the chapter section by section:

| Notebook part | Textbook |
|---|---|
| 1. Setup and constants | Table 26-1 |
| 2. $\vec E = \vec F / q_0$ and why $E$ does not depend on $q_0$ | Eq. 26-3, Fig. 26-1 |
| 3. Field of a single point charge | Eq. 26-6, Fig. 26-3 |
| 4. Two charges: $\vec F_{12} = -\vec F_{21}$ | Eq. 26-4, Fig. 26-2 |
| 5. Superposition for $N$ charges | Eq. 26-7 |
| 6. Field maps: dipole, like charges, quadrupole | — |
| 7. Field lines | — |
| 8. Motion of a test charge in the field | $\vec F = q\vec E$ |
| 9. Checks against Table 26-1 | Table 26-1 |

Everything is written with plain NumPy so you can see the physics, not a library.

## 1. Setup and constants

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.rcParams.update({"figure.dpi": 110, "font.size": 11,
                     "axes.grid": True, "grid.alpha": 0.25})

# SI constants
EPS0 = 8.8541878128e-12          # C^2 / (N m^2)
K    = 1.0 / (4 * np.pi * EPS0)  # N m^2 / C^2  -> 8.988e9
E_CHARGE = 1.602176634e-19       # C
M_ELECTRON = 9.1093837015e-31    # kg
M_PROTON   = 1.67262192369e-27   # kg
A0 = 5.29177210903e-11           # Bohr radius, m

print(f"k = 1/(4*pi*eps0) = {K:.4e} N m^2 / C^2")

## 2. $\vec E = \vec F/q_0$ — the field does not care about the test charge

Eq. 26-3 defines
$$\vec E = \frac{\vec F}{q_0}.$$

The point the book makes right after that equation: $q_0$ is a positive scalar, so $\vec E$ points
along $\vec F$, and the *value* of $\vec E$ is independent of how big $q_0$ is. Let's confirm that
numerically: put a source charge $q$ at the origin, drop several different test charges at the same
point $P$, measure the force on each with Coulomb's law, and divide.

In [ ]:
def coulomb_force(q_src, r_src, q_test, r_test):
    # Force on q_test at r_test due to q_src at r_src (2-D vectors, SI).
    d = np.asarray(r_test, float) - np.asarray(r_src, float)
    r = np.linalg.norm(d)
    return K * q_src * q_test * d / r**3


q  = 2.0e-9          # source charge, C
rP = np.array([0.03, 0.04])   # point P, 5 cm from the origin

print(f"{'q0 (C)':>12} {'|F| (N)':>14} {'|E| = F/q0 (N/C)':>20}")
for q0 in [1e-12, 1e-10, 1e-9, 1e-8]:
    F = coulomb_force(q, [0, 0], q0, rP)
    E = F / q0
    print(f"{q0:>12.0e} {np.linalg.norm(F):>14.4e} {np.linalg.norm(E):>20.6f}")

print("\nSame E every time -> E is a property of the source, not of the probe.")

That is Fig. 26-1 in numbers: panel (a) is the force $\vec F$ on $q_0$, panel (b) is $\vec E$ at the
same point $P$ — parallel, and one is just the other divided by $q_0$.

Eq. 26-5 writes this as a limit $q_0 \to 0$, because a real test charge disturbs the source
distribution. In a *calculation* (as opposed to a measurement) nothing is disturbed, so we can use
Eq. 26-3 directly — which is exactly what the code above does.

## 3. Field of a single point charge (Eq. 26-6)

$$E = \frac{1}{4\pi\epsilon_0}\frac{|q|}{r^2}$$

directed radially outward from a positive $q$, inward toward a negative $q$. Written as a vector,
$$\vec E(\vec r) = \frac{1}{4\pi\epsilon_0}\, q\, \frac{\vec r - \vec r_q}{|\vec r - \vec r_q|^3}.$$

The function below is vectorised over a grid, and includes a *softening radius*: inside it the
field is frozen, purely to stop $1/r^2$ from blowing up on a pixel that lands on the charge. It has
no physical meaning — set `soft` much smaller than any distance you care about.

In [ ]:
def E_point(q, r_q, X, Y, soft=1e-3):
    # Field of a point charge q at r_q, evaluated on grids X, Y. Returns (Ex, Ey).
    dx = X - r_q[0]
    dy = Y - r_q[1]
    r2 = dx**2 + dy**2
    r2 = np.maximum(r2, soft**2)        # softening: avoids division by zero at the charge
    r3 = r2 * np.sqrt(r2)
    return K * q * dx / r3, K * q * dy / r3


# 1/r^2 check along a radial line
r = np.linspace(0.01, 0.20, 200)
Ex, Ey = E_point(1e-9, (0, 0), r, np.zeros_like(r))
E_mag = np.hypot(Ex, Ey)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(r * 100, E_mag)
ax[0].set(xlabel="r (cm)", ylabel="E (N/C)", title="E of a 1 nC charge")
ax[1].loglog(r, E_mag)
ax[1].set(xlabel="r (m)", ylabel="E (N/C)", title="log-log: slope should be -2")

slope = np.polyfit(np.log(r), np.log(E_mag), 1)[0]
print(f"fitted slope = {slope:.4f}")
plt.tight_layout(); plt.show()

### Fig. 26-3: direction of $\vec E$ around a point charge

The book asks: *how would this figure be drawn if the charge were negative?* Run the next cell and
compare the two panels — same magnitudes, arrows reversed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.6))

for ax, q, label in zip(axes, [+2e-9, -2e-9], ["q > 0 (outward)", "q < 0 (inward)"]):
    # a few sample points P1, P2, P3, ... on a circle, as in Fig. 26-3
    theta = np.deg2rad([-70, 0, 40, 110, 200])
    px, py = 0.04 * np.cos(theta), 0.04 * np.sin(theta)
    Ex, Ey = E_point(q, (0, 0), px, py)
    scale = 0.03 / np.hypot(Ex, Ey).max()       # arrows scaled for drawing only

    ax.quiver(px, py, Ex * scale, Ey * scale, angles="xy",
              scale_units="xy", scale=1, color="crimson", width=0.008)
    ax.plot(0, 0, "o", ms=14, color="tab:red" if q > 0 else "tab:blue")
    ax.text(0, 0, "+" if q > 0 else "\u2212", ha="center", va="center",
            color="white", fontweight="bold")
    for i, (x, y) in enumerate(zip(px, py), 1):
        ax.plot(x, y, "k.", ms=5)
        ax.text(x, y - 0.006, f"$P_{i}$", ha="center", fontsize=9)
    ax.set(xlim=(-0.09, 0.09), ylim=(-0.09, 0.09), title=label)
    ax.set_aspect("equal")

plt.tight_layout(); plt.show()

## 4. Two charges and Newton's third law (Fig. 26-2, Eq. 26-4)

The chapter's argument, in four steps:

* (a) $q_1$ at $A$ sets up a field $\vec E_1$ at $B$.
* (b) $q_2$ sitting at $B$ feels $\vec F_{21} = q_2 \vec E_1$.
* (c) $q_2$ at $B$ sets up a field $\vec E_2$ at $A$.
* (d) $q_1$ feels $\vec F_{12} = q_1 \vec E_2$, and $\vec F_{12} = -\vec F_{21}$ —
  even though $\vec E_1 \neq \vec E_2$ when $q_1 \neq q_2$.

Let's verify that last line with deliberately unequal charges.

In [ ]:
q1, rA = 3.0e-9, np.array([0.00, 0.0])
q2, rB = -1.0e-9, np.array([0.05, 0.0])

E1_at_B = np.array(E_point(q1, rA, rB[0], rB[1]))   # field of q1, evaluated at B
E2_at_A = np.array(E_point(q2, rB, rA[0], rA[1]))   # field of q2, evaluated at A

F21 = q2 * E1_at_B      # Eq. 26-4: force on q2
F12 = q1 * E2_at_A      # force on q1

print(f"E1 at B = {E1_at_B} N/C   |E1| = {np.linalg.norm(E1_at_B):.3f}")
print(f"E2 at A = {E2_at_A} N/C   |E2| = {np.linalg.norm(E2_at_A):.3f}")
print(f"  -> the two fields differ by a factor {np.linalg.norm(E1_at_B)/np.linalg.norm(E2_at_A):.2f}")
print()
print(f"F21 = {F21} N")
print(f"F12 = {F12} N")
print(f"F12 + F21 = {F12 + F21}  (zero to machine precision)")
print("Third law holds even though E1 and E2 do not match.")

## 5. Superposition for $N$ charges (Eq. 26-7)

$$\vec E = \sum_{n=1}^{N} \vec E_n$$

Compute each charge's field *as if it were the only one present*, then add the vectors. This one
function carries the rest of the notebook.

In [ ]:
def E_total(charges, X, Y, soft=1e-3):
    # Superposed field of a list of point charges.
    # charges: iterable of (q, (x, y)) in SI units.
    Ex = np.zeros_like(np.asarray(X, float))
    Ey = np.zeros_like(np.asarray(Y, float))
    for q, r_q in charges:
        ex, ey = E_point(q, r_q, X, Y, soft)
        Ex += ex
        Ey += ey
    return Ex, Ey


def force_on(q_test, r_test, charges, soft=1e-6):
    # F = qE (Eq. 26-4), for a test charge placed among `charges`.
    Ex, Ey = E_total(charges, np.array(r_test[0]), np.array(r_test[1]), soft)
    return q_test * np.array([float(Ex), float(Ey)])


# sanity check: superposition vs. direct Coulomb sum
charges = [(2e-9, (-0.03, 0.0)), (-4e-9, (0.03, 0.02))]
rP, qP = (0.01, -0.02), 1e-9

F_super = force_on(qP, rP, charges)
F_direct = sum(coulomb_force(q, rq, qP, rP) for q, rq in charges)
print("superposition:", F_super)
print("direct sum:   ", F_direct)
print("agree:", np.allclose(F_super, F_direct))

## 6. Field maps

A quiver plot with every arrow the same length shows *direction*; colour carries the *magnitude*
(on a log scale, because $E$ spans orders of magnitude across the frame).

In [ ]:
def field_map(charges, extent=0.10, n=26, ax=None, title=""):
    if ax is None:
        _, ax = plt.subplots(figsize=(5.2, 5.2))
    g = np.linspace(-extent, extent, n)
    X, Y = np.meshgrid(g, g)
    Ex, Ey = E_total(charges, X, Y, soft=0.006)

    mag = np.hypot(Ex, Ey)
    u, v = Ex / mag, Ey / mag                  # unit arrows: direction only
    q = ax.quiver(X, Y, u, v, np.log10(mag), cmap="viridis",
                  pivot="mid", scale=32, width=0.004)
    plt.colorbar(q, ax=ax, shrink=0.8, label=r"$\log_{10} E$  (N/C)")

    for qc, (x, y) in charges:
        ax.plot(x, y, "o", ms=13, color="tab:red" if qc > 0 else "tab:blue", zorder=5)
        ax.text(x, y, "+" if qc > 0 else "\u2212", color="white", ha="center",
                va="center", fontweight="bold", zorder=6)
    ax.set(xlim=(-extent, extent), ylim=(-extent, extent), title=title,
           xlabel="x (m)", ylabel="y (m)")
    ax.set_aspect("equal")
    return ax


d = 0.03
configs = {
    "Single positive charge":      [(4e-9, (0, 0))],
    "Dipole  (+q, \u2212q)":       [(4e-9, (-d, 0)), (-4e-9, (d, 0))],
    "Two like charges (+q, +q)":   [(4e-9, (-d, 0)), (4e-9, (d, 0))],
    "Quadrupole":                  [(4e-9, (-d, d)), (-4e-9, (d, d)),
                                    (-4e-9, (-d, -d)), (4e-9, (d, -d))],
}

fig, axes = plt.subplots(2, 2, figsize=(11.5, 11))
for ax, (name, cfg) in zip(axes.ravel(), configs.items()):
    field_map(cfg, ax=ax, title=name)
plt.tight_layout(); plt.show()

Things worth reading off these plots:

* **Dipole** — between the charges every contribution points the same way (from $+$ to $-$), so the
  field is strongest there.
* **Two like charges** — halfway between them the two contributions cancel exactly. That null point
  is a standard HRK problem; the cell below locates it numerically.
* **Quadrupole** — far away the field falls off faster than $1/r^2$, because the charges add to zero.

In [ ]:
from scipy.optimize import brentq

like = [(4e-9, (-d, 0)), (4e-9, (d, 0))]
Ex_on_axis = lambda x: float(E_total(like, np.array(x), np.array(0.0), soft=1e-9)[0])
x_null = brentq(Ex_on_axis, -d + 1e-6, d - 1e-6)
print(f"E_x = 0 at x = {x_null:.3e} m  (midpoint, as expected by symmetry)")

# unequal charges -> null point shifts toward the weaker charge
uneq = [(4e-9, (-d, 0)), (1e-9, (d, 0))]
f = lambda x: float(E_total(uneq, np.array(x), np.array(0.0), soft=1e-9)[0])
print(f"with q2 = q1/4, null point at x = {brentq(f, -d+1e-6, d-1e-6)*100:.3f} cm")

## 7. Field lines

Field lines are curves tangent to $\vec E$ everywhere. `streamplot` draws exactly that. The
book introduces them as a picture of the field; note the rules they obey — lines start on positive
charges, end on negative ones, and never cross (if they did, $\vec E$ would have two directions at
one point).

In [ ]:
def field_lines(charges, extent=0.10, n=300, ax=None, title="", density=1.6):
    if ax is None:
        _, ax = plt.subplots(figsize=(5.4, 5.4))
    g = np.linspace(-extent, extent, n)
    X, Y = np.meshgrid(g, g)
    Ex, Ey = E_total(charges, X, Y, soft=0.004)
    mag = np.hypot(Ex, Ey)

    ax.streamplot(X, Y, Ex, Ey, color=np.log10(mag), cmap="plasma",
                  density=density, linewidth=0.9, arrowsize=0.9)
    for qc, (x, y) in charges:
        ax.plot(x, y, "o", ms=13, color="tab:red" if qc > 0 else "tab:blue", zorder=5)
        ax.text(x, y, "+" if qc > 0 else "\u2212", color="white", ha="center",
                va="center", fontweight="bold", zorder=6)
    ax.set(xlim=(-extent, extent), ylim=(-extent, extent), title=title,
           xlabel="x (m)", ylabel="y (m)")
    ax.set_aspect("equal")
    return ax


fig, axes = plt.subplots(2, 2, figsize=(11.5, 11))
for ax, (name, cfg) in zip(axes.ravel(), configs.items()):
    field_lines(cfg, ax=ax, title=name)
plt.tight_layout(); plt.show()

### Your turn

Change `my_charges` and re-run. Some setups worth trying:

* `(+q, -2q)` — an unbalanced dipole; count how many lines leave $+q$ versus enter $-2q$.
* A row of alternating charges — a crude 1-D crystal.
* A ring of equal charges — the field at the centre should vanish.

In [ ]:
my_charges = [(3e-9, (-0.04, 0.0)), (-6e-9, (0.04, 0.0))]   # <-- edit me

fig, ax = plt.subplots(1, 2, figsize=(11, 5))
field_map(my_charges, ax=ax[0], title="vectors")
field_lines(my_charges, ax=ax[1], title="field lines")
plt.tight_layout(); plt.show()

## 8. Motion of a charge in the field: $\vec F = q\vec E$, $\ \vec a = q\vec E/m$

Once you have $\vec E$, Eq. 26-4 gives the force on *any* charge placed there, and Newton's second
law gives the motion. Here an electron is launched near a dipole and its path integrated with
`solve_ivp`.

Note this is the *test-charge-as-probe* idea taken literally: we are letting the probe move, and
we are ignoring the field the electron itself creates (it does not act on itself, and we assume the
source charges are held fixed).

In [ ]:
def trajectory(charges, q, m, r0, v0, t_end, n_pts=2000, soft=2e-3):
    def rhs(t, s):
        x, y, vx, vy = s
        Ex, Ey = E_total(charges, np.array(x), np.array(y), soft)
        return [vx, vy, q * float(Ex) / m, q * float(Ey) / m]
    t_eval = np.linspace(0, t_end, n_pts)
    return solve_ivp(rhs, (0, t_end), [*r0, *v0], t_eval=t_eval,
                     rtol=1e-8, atol=1e-10, method="DOP853")


dipole = [(2e-9, (-0.02, 0.0)), (-2e-9, (0.02, 0.0))]

fig, ax = plt.subplots(figsize=(6.4, 6.4))
field_lines(dipole, extent=0.09, ax=ax, title="electron trajectories near a dipole", density=1.1)

for y0, colour in zip([0.012, 0.020, 0.030, 0.045], ["k", "dimgray", "navy", "darkgreen"]):
    sol = trajectory(dipole, -E_CHARGE, M_ELECTRON,
                     r0=(-0.085, y0), v0=(2.0e6, 0.0), t_end=1.0e-7)
    ax.plot(sol.y[0], sol.y[1], lw=2.2, color=colour, zorder=7,
            label=f"$y_0$ = {y0*100:.1f} cm")

ax.legend(loc="lower left", fontsize=9)
plt.tight_layout(); plt.show()

Try flipping the sign of the charge (use `+E_CHARGE` and `M_PROTON` for a proton) and watch the
deflection reverse — the field is unchanged, only $q$ in $\vec F = q\vec E$ flips.

### Uniform field: the parallel-plate case

For a uniform $\vec E$ the motion is projectile-like — constant acceleration along the field. This
is the deflection used in the TV-tube example in Table 26-1 ($E \approx 10^5$ N/C).

In [ ]:
E_uniform = np.array([0.0, -1.0e5])       # N/C, pointing -y
v0 = np.array([3.0e7, 0.0])               # m/s
L = 0.04                                  # plate length, m
t = np.linspace(0, L / v0[0], 300)

a = (-E_CHARGE) * E_uniform / M_ELECTRON  # electron
x = v0[0] * t
y = 0.5 * a[1] * t**2

plt.figure(figsize=(6.4, 3.6))
plt.plot(x * 100, y * 100, lw=2)
plt.xlabel("x (cm)"); plt.ylabel("deflection y (cm)")
plt.title("Electron in a uniform field, E = $10^5$ N/C")
print(f"acceleration = {a[1]:.3e} m/s^2")
print(f"deflection after {L*100:.0f} cm of travel: {y[-1]*1000:.3f} mm")
plt.tight_layout(); plt.show()

## 9. Checks against Table 26-1

The table lists $E$ at the electron's average radius in hydrogen as $5\times10^{11}$ N/C, and at the
surface of a uranium nucleus as $3\times10^{21}$ N/C. Both follow from Eq. 26-6 — good practice at
getting orders of magnitude right.

In [ ]:
def E_of_point_charge(q, r):
    return K * abs(q) / r**2

rows = [
    ("Hydrogen, at Bohr radius", E_of_point_charge(E_CHARGE, A0), 5e11),
    ("Uranium nucleus surface (Z=92, R=7.4 fm)",
     E_of_point_charge(92 * E_CHARGE, 7.4e-15), 3e21),
]

print(f"{'situation':<44}{'computed (N/C)':>18}{'Table 26-1':>14}")
for name, val, book in rows:
    print(f"{name:<44}{val:>18.3e}{book:>14.0e}")

# Inverse question: how close to a 1 nC charge before air breaks down (3e6 N/C)?
r_break = np.sqrt(K * 1e-9 / 3e6)
print(f"\nAir breaks down within {r_break*1000:.3f} mm of a 1 nC point charge.")

## Where to take this next

* **Continuous distributions** (the "Later we will generalize..." line in 26-3): replace the sum in
  `E_total` with a numerical integral — a charged rod, ring, or disc discretised into many small
  $dq$. Compare against the closed-form results in §26-4 and §26-5.
* **Dipole in an external field** (§26-6): compute the torque $\vec\tau = \vec p \times \vec E$ and
  animate the oscillation.
* **Gauss's law** (Ch. 27): integrate $\vec E \cdot d\vec A$ numerically over a closed surface around
  your charges and check it equals $q_{\rm enc}/\epsilon_0$. That is a genuinely satisfying
  numerical check and a natural next notebook.